In [ ]:
import gffutils

gffutils.constants.always_return_list = True

#Creates a database file of the GTF/GFF3 file provided
db = gffutils.create_db("/home/marc/data/rna_seq_foxy/genome/GCF_000149955.1/genomic.gtf", "ASM14995v2_db")


In [68]:
import pandas as pd
from Bio import SeqIO

positive_DEG_df = pd.read_csv("/home/marc/projects/rna_seq_workflow/results/DEG_analysis/FOXG_ASM14995v2_ranked_positive_genes.csv")
product_annotation_df = pd.read_csv("/home/marc/projects/rna_seq_workflow/results/feature_counts/FOXG_ASM14995v2_product_annotation.csv")


positive_DEG_df = positive_DEG_df.set_index("symbol").join(product_annotation_df.set_index("Geneid"))
positive_DEG_df.head()

,log2FC_shrinked,product,exon_number
symbol,,,
FOXG_12350,8.596137,cytochrome P450 55A1,8;7;6;5;4;3;2;1
FOXG_03770,7.515708,nitrate reductase (NADH),4;3;3;3;2;2;2;1;1;1
FOXG_09804,7.221403,hypothetical protein,3;2;1
FOXG_20229,6.513134,hypothetical protein,1;1;2;2
FOXG_11545,6.298982,"cytochrome P450, family 51 (sterol 14-demethyl...",1;1;1;1;1;2;2;2;3;2;2;3


In [69]:
positive_DEG_df_filt = positive_DEG_df[positive_DEG_df.iloc[:, 0] >= 2]

positive_DEG_df_filt.tail()

,log2FC_shrinked,product,exon_number
symbol,,,
FOXG_13788,2.050642,hypothetical protein,1;2
FOXG_13064,2.040726,hypothetical protein,1;1;1;2;2;2;3;4;3
FOXG_09892,2.013350,hypothetical protein,3;2;1
FOXG_11427,2.009088,"MFS transporter, SHS family, lactate transporter",4;3;2;1
FOXG_06140,2.000459,uroporphyrin-III C-methyltransferase/precorrin...,1;2


In [70]:

#Returns string instead of list
gffutils.constants.always_return_list = False
gene_ids = positive_DEG_df_filt.index.tolist()
protein_list = []
all_protein_ids = []
protein_dict = {}

#Builds a dict with gene_ids as key and protein_ids as value
for gene_id in gene_ids:
    #Gene entry in GTF
    gene = db[gene_id]
    #The CDS entry contains the protein_id so here for every CDS feature ("children") of a gene of interest 
    #all corresponding protein_ids are appended to a list
    for i in db.children(gene, featuretype="CDS"):
        protein_list.append(i["protein_id"])
    
    #To remove duplicates the list is turned into a set
    protein_dict[gene_id] = set(protein_list)
    #The list for the current gene_id is emptied, so that it can be filled with the protein_ids for the next gene
    protein_list = []

protein_df = pd.DataFrame([protein_dict])
protein_df = protein_df.transpose().rename(columns={0:"protein_ids"})
positive_DEG_df_filt = positive_DEG_df_filt.join(protein_df)

#positive_DEG_df_filt.to_csv("")

#Flattens nested list of all protein_ids of interest
ids_to_extract = [item for sublist in list(protein_dict.values()) for item in sublist]

positive_DEG_df_filt.head()

,log2FC_shrinked,product,exon_number,protein_ids
symbol,,,,
FOXG_12350,8.596137,cytochrome P450 55A1,8;7;6;5;4;3;2;1,{XP_018250912.1}
FOXG_03770,7.515708,nitrate reductase (NADH),4;3;3;3;2;2;2;1;1;1,"{XP_018238161.1, XP_018238160.1, XP_018238162.1}"
FOXG_09804,7.221403,hypothetical protein,3;2;1,{XP_018247215.1}
FOXG_20229,6.513134,hypothetical protein,1;1;2;2,"{XP_018247608.1, XP_018247609.1}"
FOXG_11545,6.298982,"cytochrome P450, family 51 (sterol 14-demethyl...",1;1;1;1;1;2;2;2;3;2;2;3,"{XP_018249823.1, XP_018249824.1, XP_018249825...."


In [ ]:
from Bio import SeqIO

# Parse the fasta file
records = SeqIO.parse("/home/marc/data/rna_seq_foxy/genome/GCF_000149955.1/protein.faa", "fasta")

# Write selected sequences to a new fasta file
with open("positive_DEG_proteins.fasta", "w") as output_handle:
    SeqIO.write((record for record in records if record.id in ids_to_extract), output_handle, "fasta")
